# Évaluation stricte de la méthode du notebook 05

Ce notebook vérifie si le résultat exploratoire du notebook 05 tient quand on applique une validation vraiment stricte, groupée par sujet et sans fuite de sélection.

## 0. Configuration

On fixe volontairement la famille de méthode du notebook 05 : `HistGradientBoosting`, segments de 300 frappes, features temporelles uniquement, et PSO. La différence importante est que PSO est relancé dans chaque fold externe, uniquement sur les sujets d’entraînement.

In [1]:
from pathlib import Path
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pyswarms.discrete import BinaryPSO
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
OUTER_SPLITS = 5
INNER_SPLITS = 5
SEGMENT_MIN_LEN = 300
WINDOW = 300
STRIDE = 150
MODEL_NAME = "histgb"
PSO_PARTICLES = 18
PSO_ITERS = 12
SAVE_FINAL_MODEL = False

ROOT = Path.cwd()
if not (ROOT / "data" / "neuroqwerty-mit-csxpd-dataset-1.0.0").exists() and ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_ROOT = ROOT / "data" / "neuroqwerty-mit-csxpd-dataset-1.0.0"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

NOTEBOOK05_PSO_TIMING_FEATURES = [
    "duration_sec",
    "mean_hold",
    "std_hold",
    "median_hold",
    "iqr_hold",
    "q10_hold",
    "q90_hold",
    "kurt_hold",
    "cv_hold",
    "mean_flight",
    "std_flight",
    "q90_flight",
    "skew_flight",
    "kurt_flight",
]


def format_seconds(seconds):
    seconds = max(0, float(seconds))
    hours, remainder = divmod(int(seconds), 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"

print("ROOT", ROOT)
print("Strict audit of notebook 05: HistGB + timing-only + PSO, min_len=300")


ROOT /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Hackaton/parkinson-detection
Strict audit of notebook 05: HistGB + timing-only + PSO, min_len=300


## 1. Chargement et nettoyage

In [2]:
KEY_COLUMNS = ["key", "hold_time", "release_time", "press_time"]
MOUSE_RE = re.compile(r'"?mouse.+"?', re.I)
LONG_META_RE = re.compile(r'"?(Shift.+|Alt.+|Control.+)"?', re.I)
BACKSPACE_RE = re.compile(r'"?BackSpace"?', re.I)
PUNCT_OR_SPACE_RE = re.compile(r'"?(space|comma|period|semicolon|slash|minus|equal|apostrophe|Return)"?', re.I)
LEFT_KEYS = set("qwertasdfgzxcvb")
RIGHT_KEYS = set("yuiophjklnm")


def load_raw(path):
    df = pd.read_csv(path, header=None, names=KEY_COLUMNS)
    df["key"] = df["key"].astype(str).str.strip().str.replace('"', "", regex=False)
    for column in ["hold_time", "release_time", "press_time"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def clean(df):
    d = df.dropna(subset=["hold_time", "release_time", "press_time"]).copy()
    key = d["key"].astype(str)
    keep = ~key.str.match(MOUSE_RE) & ~key.str.match(LONG_META_RE) & ~key.str.match(BACKSPACE_RE)
    d = d.loc[keep]
    d = d[(d.press_time > 0) & (d.release_time > 0) & (d.hold_time.between(0, 5))]
    d = d.sort_values("press_time").reset_index(drop=True)
    d["flight_time"] = d.press_time.diff()
    d.loc[d.flight_time < 0, "flight_time"] = np.nan
    d["is_space_punct"] = d.key.str.match(PUNCT_OR_SPACE_RE).astype(int)
    d["hand_left"] = d.key.str.lower().str[:1].isin(LEFT_KEYS).astype(int)
    d["hand_right"] = d.key.str.lower().str[:1].isin(RIGHT_KEYS).astype(int)
    d["hand_switch"] = (d.hand_left.diff().abs().fillna(0) > 0).astype(int)
    return d


def load_sessions():
    rows = []
    raws = {}
    for dataset in ["MIT-CS1PD", "MIT-CS2PD"]:
        gt = pd.read_csv(DATA_ROOT / dataset / f"GT_DataPD_{dataset}.csv")
        raw_dir = DATA_ROOT / dataset / f"data_{dataset}"
        for _, subject in gt.iterrows():
            for file_col in [column for column in gt.columns if column.startswith("file_")]:
                filename = subject.get(file_col)
                if pd.isna(filename) or not str(filename).strip():
                    continue
                session_uid = f"{dataset}_{int(subject.pID)}_{file_col}"
                raw_path = raw_dir / str(filename)
                cleaned = clean(load_raw(raw_path))
                raws[session_uid] = cleaned
                rows.append({
                    "session_uid": session_uid,
                    "dataset": dataset,
                    "pID": int(subject.pID),
                    "session_file": str(filename),
                    "session_id": file_col,
                    "label": int(bool(subject["gt"])),
                    "n_keys": len(cleaned),
                })
    return pd.DataFrame(rows), raws


sessions, raw_sessions = load_sessions()
print("sessions", len(sessions), "subjects", sessions.pID.nunique())
display(sessions.groupby("label").agg(sessions=("label", "size"), subjects=("pID", "nunique")))


sessions 116 subjects 85


,sessions,subjects
label,,
0,56,43
1,60,42


## 2. Features segmentées avec le protocole du notebook 05

In [3]:
def safe_div(a, b):
    return np.nan if b is None or pd.isna(b) or abs(b) < 1e-12 else a / b


def entropy_binary(p):
    if pd.isna(p) or p <= 0 or p >= 1:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def agg_features(d):
    hold = d.hold_time.dropna()
    flight = d.flight_time.dropna()
    duration = d.release_time.max() - d.press_time.min() if len(d) else np.nan
    out = {
        "n_keystrokes": len(d),
        "duration_sec": duration,
        "keys_per_min": safe_div(len(d) * 60, duration),
        "mean_hold": hold.mean(),
        "std_hold": hold.std(),
        "median_hold": hold.median(),
        "iqr_hold": hold.quantile(0.75) - hold.quantile(0.25),
        "q10_hold": hold.quantile(0.1),
        "q90_hold": hold.quantile(0.9),
        "skew_hold": hold.skew(),
        "kurt_hold": hold.kurt(),
        "cv_hold": safe_div(hold.std(), hold.mean()),
        "mean_flight": flight.mean(),
        "std_flight": flight.std(),
        "median_flight": flight.median(),
        "iqr_flight": flight.quantile(0.75) - flight.quantile(0.25),
        "q10_flight": flight.quantile(0.1),
        "q90_flight": flight.quantile(0.9),
        "skew_flight": flight.skew(),
        "kurt_flight": flight.kurt(),
        "cv_flight": safe_div(flight.std(), flight.mean()),
        "hold_to_flight": safe_div(hold.mean(), flight.mean()),
        "long_hold_rate": (hold > hold.quantile(0.9)).mean() if len(hold) > 5 else np.nan,
        "long_flight_rate": (flight > 1.0).mean() if len(flight) > 5 else np.nan,
        "space_punct_rate": d.is_space_punct.mean(),
        "left_rate": d.hand_left.mean(),
        "right_rate": d.hand_right.mean(),
        "hand_switch_rate": d.hand_switch.mean(),
    }
    out["hand_entropy"] = entropy_binary(out["left_rate"])
    return out


def build_segment_features(window=300, stride=150, min_len=300):
    rows = []
    for _, session in sessions.iterrows():
        d = raw_sessions[session.session_uid]
        starts = list(range(0, max(len(d) - min_len + 1, 0), stride))
        for segment_id, start in enumerate(starts):
            segment = d.iloc[start:min(start + window, len(d))]
            if len(segment) < min_len:
                continue
            rows.append({
                **session.to_dict(),
                "segment_id": segment_id,
                "segment_start": start,
                "segment_len": len(segment),
                **agg_features(segment),
            })
    return pd.DataFrame(rows)



segment_features = build_segment_features(window=WINDOW, stride=STRIDE, min_len=SEGMENT_MIN_LEN)
print("segment_features", segment_features.shape, "sessions represented", segment_features.session_uid.nunique())
display(segment_features.groupby("label").agg(segments=("label", "size"), sessions=("session_uid", "nunique"), subjects=("pID", "nunique")))


segment_features (951, 39) sessions represented 115


,segments,sessions,subjects
label,,,
0,494,55,42
1,457,60,42


## 3. Jeux de features

On compare trois niveaux : toutes les features temporelles, les 14 features sélectionnées par le notebook 05, et PSO relancé proprement dans chaque fold externe.

In [4]:

META_COLS = ["session_uid", "dataset", "pID", "session_file", "session_id", "label", "gt", "n_keys", "segment_id", "segment_start", "segment_len"]
all_features = [column for column in segment_features.columns if column not in META_COLS]
all_features = [column for column in all_features if pd.api.types.is_numeric_dtype(segment_features[column])]
layout_features = ["space_punct_rate", "left_rate", "right_rate", "hand_switch_rate", "hand_entropy"]
timing_features = [feature for feature in all_features if feature not in layout_features]

missing_notebook05 = [feature for feature in NOTEBOOK05_PSO_TIMING_FEATURES if feature not in timing_features]
if missing_notebook05:
    raise ValueError(f"Missing notebook 05 features: {missing_notebook05}")

print("all_features", len(all_features), all_features)
print("timing_features", len(timing_features), timing_features)
print("notebook05_pso_timing_features", len(NOTEBOOK05_PSO_TIMING_FEATURES), NOTEBOOK05_PSO_TIMING_FEATURES)


all_features 29 ['n_keystrokes', 'duration_sec', 'keys_per_min', 'mean_hold', 'std_hold', 'median_hold', 'iqr_hold', 'q10_hold', 'q90_hold', 'skew_hold', 'kurt_hold', 'cv_hold', 'mean_flight', 'std_flight', 'median_flight', 'iqr_flight', 'q10_flight', 'q90_flight', 'skew_flight', 'kurt_flight', 'cv_flight', 'hold_to_flight', 'long_hold_rate', 'long_flight_rate', 'space_punct_rate', 'left_rate', 'right_rate', 'hand_switch_rate', 'hand_entropy']
timing_features 24 ['n_keystrokes', 'duration_sec', 'keys_per_min', 'mean_hold', 'std_hold', 'median_hold', 'iqr_hold', 'q10_hold', 'q90_hold', 'skew_hold', 'kurt_hold', 'cv_hold', 'mean_flight', 'std_flight', 'median_flight', 'iqr_flight', 'q10_flight', 'q90_flight', 'skew_flight', 'kurt_flight', 'cv_flight', 'hold_to_flight', 'long_hold_rate', 'long_flight_rate']
notebook05_pso_timing_features 14 ['duration_sec', 'mean_hold', 'std_hold', 'median_hold', 'iqr_hold', 'q10_hold', 'q90_hold', 'kurt_hold', 'cv_hold', 'mean_flight', 'std_flight', 'q90

## 4. Fonctions modèle et validation

In [5]:
def model_factory(name):
    if name == "histgb":
        return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.04, l2_regularization=0.1, random_state=RANDOM_STATE)
    raise ValueError(f"Unknown model: {name}")


def make_pipeline(model_name):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model_factory(model_name)),
    ])


def aggregate_segment_predictions(table, valid_idx, probabilities, threshold):
    tmp = table.iloc[valid_idx][["session_uid", "pID", "label"]].copy()
    tmp["proba"] = probabilities
    agg = tmp.groupby("session_uid").agg(
        label=("label", "first"),
        pID=("pID", "first"),
        proba_mean=("proba", "mean"),
        n_segments=("proba", "size"),
    ).reset_index()
    agg["pred"] = (agg.proba_mean >= threshold).astype(int)
    return agg


def metrics_from_predictions(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_binary": f1_score(y_true, y_pred),
        "precision_binary": precision_score(y_true, y_pred, zero_division=0),
        "recall_binary": recall_score(y_true, y_pred, zero_division=0),
    }


def evaluate_on_outer_fold(table, features, model_name, train_subjects, test_subjects, threshold):
    train_table = table[table.pID.isin(train_subjects)].copy()
    test_table = table[table.pID.isin(test_subjects)].copy()
    clf = make_pipeline(model_name)
    clf.fit(train_table[features], train_table.label.astype(int))
    probabilities = clf.predict_proba(test_table[features])[:, 1]
    agg = aggregate_segment_predictions(test_table, np.arange(len(test_table)), probabilities, threshold=threshold)
    return agg, metrics_from_predictions(agg.label, agg.pred)


def inner_cv_oof(table, features, model_name, train_subjects, n_splits=INNER_SPLITS):
    train_table = table[table.pID.isin(train_subjects)].copy()
    X = train_table[features]
    y = train_table.label.astype(int)
    groups = train_table.pID.astype(str)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = []
    fold_rows = []
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), 1):
        clf = make_pipeline(model_name)
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        probabilities = clf.predict_proba(X.iloc[valid_idx])[:, 1]
        agg = aggregate_segment_predictions(train_table, valid_idx, probabilities, threshold=0.5)
        fold_rows.append({"inner_fold": fold, **metrics_from_predictions(agg.label, agg.pred)})
        oof.append(agg.assign(inner_fold=fold))
    return pd.concat(oof, ignore_index=True), pd.DataFrame(fold_rows)


def choose_threshold(oof, thresholds=np.linspace(0.2, 0.8, 61)):
    rows = []
    for threshold in thresholds:
        pred = (oof.proba_mean >= threshold).astype(int)
        rows.append({"threshold": threshold, **metrics_from_predictions(oof.label, pred)})
    threshold_df = pd.DataFrame(rows)
    best = threshold_df.sort_values(["f1_macro", "balanced_accuracy"], ascending=False).iloc[0]
    return float(best.threshold), threshold_df


## 5. PSO imbriqué

In [6]:
def pso_select_features_nested(table, candidate_features, model_name, train_subjects, n_particles=PSO_PARTICLES, iters=PSO_ITERS, feature_penalty=0.015, progress_label=""):
    candidate_features = list(candidate_features)
    cache = {}
    pso_start = time.time()
    objective_calls = {"count": 0}

    def objective(masks):
        costs = []
        objective_calls["count"] += 1
        call_start = time.time()
        new_masks = 0
        for mask in masks:
            selected = [feature for feature, keep in zip(candidate_features, mask.astype(bool)) if keep]
            if not selected:
                costs.append(1.0)
                continue
            key = tuple(selected)
            if key not in cache:
                new_masks += 1
                oof, _ = inner_cv_oof(table, selected, model_name, train_subjects)
                threshold, _ = choose_threshold(oof)
                pred = (oof.proba_mean >= threshold).astype(int)
                mean_f1 = f1_score(oof.label, pred, average="macro")
                penalty = feature_penalty * (len(selected) / len(candidate_features))
                cache[key] = 1.0 - mean_f1 + penalty
            costs.append(cache[key])
        print(
            f"      PSO {progress_label} call {objective_calls['count']}/{iters}: "
            f"particles={len(masks)}, new_masks={new_masks}, cache={len(cache)}, "
            f"elapsed={format_seconds(time.time() - pso_start)}, call={format_seconds(time.time() - call_start)}",
            flush=True,
        )
        return np.array(costs)

    print(
        f"    PSO start {progress_label}: features={len(candidate_features)}, particles={n_particles}, iters={iters}",
        flush=True,
    )
    options = {"c1": 0.8, "c2": 0.8, "w": 0.7, "k": min(8, n_particles - 1), "p": 2}
    optimizer = BinaryPSO(n_particles=n_particles, dimensions=len(candidate_features), options=options)
    best_cost, best_pos = optimizer.optimize(objective, iters=iters, verbose=False)
    selected = [feature for feature, keep in zip(candidate_features, best_pos.astype(bool)) if keep]
    print(
        f"    PSO end {progress_label}: selected={len(selected)}, best_cost={best_cost:.4f}, "
        f"elapsed={format_seconds(time.time() - pso_start)}",
        flush=True,
    )
    return selected, float(best_cost)


## 6. Évaluation stricte ciblée

Cette cellule est longue. Elle répond à la question : est-ce que `PSO timing-only` du notebook 05 généralise quand la sélection PSO est faite uniquement dans les données d’entraînement du fold externe ?

In [7]:

def evaluate_fixed_feature_set(features, label):
    """Evaluate a fixed feature set with outer grouped CV and inner threshold tuning only."""
    y_subject = sessions.groupby("pID").label.first()
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    oof_rows = []
    subjects = y_subject.index.to_numpy()
    labels = y_subject.to_numpy()
    for outer_fold, (train_subject_idx, test_subject_idx) in enumerate(outer_cv.split(subjects, labels, groups=subjects), 1):
        train_subjects = set(subjects[train_subject_idx])
        test_subjects = set(subjects[test_subject_idx])
        print(f"{label} | outer fold {outer_fold}/{OUTER_SPLITS}: train_subjects={len(train_subjects)} test_subjects={len(test_subjects)}", flush=True)
        inner_oof, _ = inner_cv_oof(segment_features, features, MODEL_NAME, train_subjects, n_splits=INNER_SPLITS)
        threshold, _ = choose_threshold(inner_oof)
        agg, metrics = evaluate_on_outer_fold(segment_features, features, MODEL_NAME, train_subjects, test_subjects, threshold)
        rows.append({
            "experiment": label,
            "outer_fold": outer_fold,
            "n_features": len(features),
            "threshold": threshold,
            **metrics,
        })
        oof_rows.append(agg.assign(experiment=label, outer_fold=outer_fold))
        print(f"  threshold={threshold:.2f} metrics={metrics}", flush=True)
    return pd.DataFrame(rows), pd.concat(oof_rows, ignore_index=True)


def evaluate_nested_pso_timing():
    """Evaluate notebook 05 methodology without leakage: PSO is rerun inside each outer training fold."""
    y_subject = sessions.groupby("pID").label.first()
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    oof_rows = []
    selected_rows = []
    subjects = y_subject.index.to_numpy()
    labels = y_subject.to_numpy()
    start = time.time()
    for outer_fold, (train_subject_idx, test_subject_idx) in enumerate(outer_cv.split(subjects, labels, groups=subjects), 1):
        train_subjects = set(subjects[train_subject_idx])
        test_subjects = set(subjects[test_subject_idx])
        print("\n" + "=" * 96, flush=True)
        print(f"nested_pso_timing | outer fold {outer_fold}/{OUTER_SPLITS}: train_subjects={len(train_subjects)} test_subjects={len(test_subjects)}", flush=True)
        selected_features, best_cost = pso_select_features_nested(
            segment_features,
            timing_features,
            MODEL_NAME,
            train_subjects,
            n_particles=PSO_PARTICLES,
            iters=PSO_ITERS,
            feature_penalty=0.015,
            progress_label=f"outer={outer_fold} notebook05_histgb_timing",
        )
        inner_oof, _ = inner_cv_oof(segment_features, selected_features, MODEL_NAME, train_subjects, n_splits=INNER_SPLITS)
        threshold, _ = choose_threshold(inner_oof)
        agg, metrics = evaluate_on_outer_fold(segment_features, selected_features, MODEL_NAME, train_subjects, test_subjects, threshold)
        rows.append({
            "experiment": "nested_pso_timing_histgb",
            "outer_fold": outer_fold,
            "n_features": len(selected_features),
            "threshold": threshold,
            "best_cost": best_cost,
            **metrics,
        })
        selected_rows.append({
            "outer_fold": outer_fold,
            "n_features": len(selected_features),
            "threshold": threshold,
            "features": selected_features,
        })
        oof_rows.append(agg.assign(experiment="nested_pso_timing_histgb", outer_fold=outer_fold))
        print(f"selected={len(selected_features)} threshold={threshold:.2f} metrics={metrics}", flush=True)
        print(f"elapsed={format_seconds(time.time() - start)}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(selected_rows), pd.concat(oof_rows, ignore_index=True)

baseline_timing_results, baseline_timing_oof = evaluate_fixed_feature_set(timing_features, "timing_all_histgb")
notebook05_static_results, notebook05_static_oof = evaluate_fixed_feature_set(NOTEBOOK05_PSO_TIMING_FEATURES, "notebook05_static_features_histgb")
nested_pso_results, nested_pso_selected, nested_pso_oof = evaluate_nested_pso_timing()

all_results = pd.concat([baseline_timing_results, notebook05_static_results, nested_pso_results], ignore_index=True)
display(all_results)
display(all_results.groupby("experiment")[["accuracy", "balanced_accuracy", "f1_macro", "f1_binary", "precision_binary", "recall_binary"]].agg(["mean", "std"]).round(3))
display(nested_pso_selected)


timing_all_histgb | outer fold 1/5: train_subjects=68 test_subjects=17
  threshold=0.60 metrics={'accuracy': 0.7272727272727273, 'balanced_accuracy': 0.7272727272727273, 'f1_macro': 0.7272727272727273, 'f1_binary': 0.7272727272727273, 'precision_binary': 0.7272727272727273, 'recall_binary': 0.7272727272727273}
timing_all_histgb | outer fold 2/5: train_subjects=68 test_subjects=17
  threshold=0.20 metrics={'accuracy': 0.625, 'balanced_accuracy': 0.6048951048951049, 'f1_macro': 0.5901328273244781, 'f1_binary': 0.7096774193548387, 'precision_binary': 0.6111111111111112, 'recall_binary': 0.8461538461538461}
timing_all_histgb | outer fold 3/5: train_subjects=68 test_subjects=17
  threshold=0.55 metrics={'accuracy': 0.6153846153846154, 'balanced_accuracy': 0.6011904761904762, 'f1_macro': 0.59375, 'f1_binary': 0.5, 'precision_binary': 0.625, 'recall_binary': 0.4166666666666667}
timing_all_histgb | outer fold 4/5: train_subjects=69 test_subjects=16
  threshold=0.67 metrics={'accuracy': 0.63636

,experiment,outer_fold,n_features,threshold,accuracy,balanced_accuracy,f1_macro,f1_binary,precision_binary,recall_binary,best_cost
0,timing_all_histgb,1,24,0.60,0.727273,0.727273,0.727273,0.727273,0.727273,0.727273,NaN
1,timing_all_histgb,2,24,0.20,0.625000,0.604895,0.590133,0.709677,0.611111,0.846154,NaN
2,timing_all_histgb,3,24,0.55,0.615385,0.601190,0.593750,0.500000,0.625000,0.416667,NaN
3,timing_all_histgb,4,24,0.67,0.636364,0.625000,0.623932,0.692308,0.642857,0.750000,NaN
4,timing_all_histgb,5,24,0.44,0.761905,0.750000,0.752941,0.800000,0.769231,0.833333,NaN
5,notebook05_static_features_histgb,1,14,0.70,0.681818,0.681818,0.675789,0.631579,0.750000,0.545455,NaN
6,notebook05_static_features_histgb,2,14,0.38,0.708333,0.695804,0.695100,0.758621,0.687500,0.846154,NaN
7,notebook05_static_features_histgb,3,14,0.52,0.576923,0.571429,0.571214,0.521739,0.545455,0.500000,NaN
8,notebook05_static_features_histgb,4,14,0.51,0.681818,0.658333,0.645977,0.758621,0.647059,0.916667,NaN
9,notebook05_static_features_histgb,5,14,0.68,0.619048,0.638889,0.618182,0.600000,0.750000,0.500000,NaN


accuracy        balanced_accuracy         \
                                      mean    std              mean    std   
experiment                                                                   
nested_pso_timing_histgb             0.687  0.037             0.675  0.042   
notebook05_static_features_histgb    0.654  0.054             0.649  0.049   
timing_all_histgb                    0.673  0.067             0.662  0.071   

                                  f1_macro        f1_binary         \
                                      mean    std      mean    std   
experiment                                                           
nested_pso_timing_histgb             0.672  0.046     0.711  0.049   
notebook05_static_features_histgb    0.641  0.049     0.654  0.103   
timing_all_histgb                    0.658  0.077     0.686  0.112   

                                  precision_binary        recall_binary         
                                              mean    std          mean    std  
experiment                                                                      
nested_pso_timing_histgb                     0.693  0.060         0.747  0.126  
notebook05_static_features_histgb            0.676  0.085         0.662  0.203  
timing_all_histgb                            0.675  0.069         0.715  0.174

,outer_fold,n_features,threshold,features
0,1,8,0.80,"[mean_hold, std_hold, median_hold, q10_hold, s..."
1,2,18,0.22,"[n_keystrokes, duration_sec, keys_per_min, iqr..."
2,3,12,0.55,"[median_hold, iqr_hold, cv_hold, mean_flight, ..."
3,4,8,0.44,"[keys_per_min, std_hold, kurt_hold, std_flight..."
4,5,10,0.38,"[keys_per_min, std_hold, q90_hold, kurt_hold, ..."


## 7. Résultats agrégés

In [8]:

all_oof = pd.concat([baseline_timing_oof, notebook05_static_oof, nested_pso_oof], ignore_index=True)

for experiment, data in all_oof.groupby("experiment"):
    print("\n" + "=" * 80)
    print(experiment)
    print(classification_report(data.label, data.pred, target_names=["Contrôle", "Parkinson"]))
    cm = confusion_matrix(data.label, data.pred)
    display(pd.DataFrame(cm, index=["Réel contrôle", "Réel Parkinson"], columns=["Prédit contrôle", "Prédit Parkinson"]))

summary = all_results.groupby("experiment")[["accuracy", "balanced_accuracy", "f1_macro", "f1_binary", "precision_binary", "recall_binary"]].agg(["mean", "std"]).round(3)
display(summary)



nested_pso_timing_histgb
              precision    recall  f1-score   support

    Contrôle       0.69      0.62      0.65        55
   Parkinson       0.68      0.75      0.71        60

    accuracy                           0.69       115
   macro avg       0.69      0.68      0.68       115
weighted avg       0.69      0.69      0.69       115



,Prédit contrôle,Prédit Parkinson
Réel contrôle,34,21
Réel Parkinson,15,45



notebook05_static_features_histgb
              precision    recall  f1-score   support

    Contrôle       0.64      0.64      0.64        55
   Parkinson       0.67      0.67      0.67        60

    accuracy                           0.65       115
   macro avg       0.65      0.65      0.65       115
weighted avg       0.65      0.65      0.65       115



,Prédit contrôle,Prédit Parkinson
Réel contrôle,35,20
Réel Parkinson,20,40



timing_all_histgb
              precision    recall  f1-score   support

    Contrôle       0.67      0.62      0.64        55
   Parkinson       0.67      0.72      0.69        60

    accuracy                           0.67       115
   macro avg       0.67      0.67      0.67       115
weighted avg       0.67      0.67      0.67       115



,Prédit contrôle,Prédit Parkinson
Réel contrôle,34,21
Réel Parkinson,17,43


accuracy        balanced_accuracy         \
                                      mean    std              mean    std   
experiment                                                                   
nested_pso_timing_histgb             0.687  0.037             0.675  0.042   
notebook05_static_features_histgb    0.654  0.054             0.649  0.049   
timing_all_histgb                    0.673  0.067             0.662  0.071   

                                  f1_macro        f1_binary         \
                                      mean    std      mean    std   
experiment                                                           
nested_pso_timing_histgb             0.672  0.046     0.711  0.049   
notebook05_static_features_histgb    0.641  0.049     0.654  0.103   
timing_all_histgb                    0.658  0.077     0.686  0.112   

                                  precision_binary        recall_binary         
                                              mean    std          mean    std  
experiment                                                                      
nested_pso_timing_histgb                     0.693  0.060         0.747  0.126  
notebook05_static_features_histgb            0.676  0.085         0.662  0.203  
timing_all_histgb                            0.675  0.069         0.715  0.174

## 8. Export optionnel

Ne pas exporter automatiquement avant d’avoir relu les résultats. Le but de ce notebook est d’abord de vérifier la robustesse du notebook 05.

In [9]:

if SAVE_FINAL_MODEL:
    # Export only after manually reviewing the strict results.
    all_subjects = set(sessions.pID.unique())
    final_features, _ = pso_select_features_nested(segment_features, timing_features, MODEL_NAME, all_subjects)
    final_oof, _ = inner_cv_oof(segment_features, final_features, MODEL_NAME, all_subjects, n_splits=OUTER_SPLITS)
    final_threshold, _ = choose_threshold(final_oof)
    final_model = make_pipeline(MODEL_NAME)
    final_model.fit(segment_features[final_features], segment_features.label.astype(int))
    artifact = {
        "pipeline": final_model,
        "features": final_features,
        "model": "HistGB",
        "level": "segment_mean_agg",
        "threshold": float(final_threshold),
        "min_segment_len": SEGMENT_MIN_LEN,
        "feature_mode": "timing",
        "selection_mode": "pso",
        "note": "Strict audit of notebook 05 methodology: HistGB + timing-only + nested PSO.",
    }
    path = MODEL_DIR / "keyboard_dynamics_neuroqwerty_notebook05_strict_pso_timing.joblib"
    joblib.dump(artifact, path)
    print(path)
else:
    print("No model exported. Set SAVE_FINAL_MODEL=True after reviewing the strict audit.")


No model exported. Set SAVE_FINAL_MODEL=True after reviewing the strict audit.


## 9. Analyse des résultats

Le notebook 07 vérifie la méthode du notebook 05 avec une validation plus stricte. Dans le notebook 05, la variante `PSO timing-only` obtenait environ `0.782` de F1 macro. Ici, la version stricte `nested_pso_timing_histgb` obtient `0.672 ± 0.046` de F1 macro.

La baisse vient principalement du fait que PSO est maintenant relancé dans chaque fold externe, uniquement sur les sujets d'entraînement. Les sujets de test du fold externe ne servent donc ni à sélectionner les features, ni à choisir le seuil. Dans le notebook 05, PSO était utilisé de manière exploratoire sur la validation groupée globale, ce qui pouvait favoriser un masque de features particulièrement adapté aux folds observés.

Comparaison stricte :

| Variante | F1 macro | Balanced accuracy | Rappel Parkinson |
|---|---:|---:|---:|
| `nested_pso_timing_histgb` | `0.672 ± 0.046` | `0.675 ± 0.042` | `0.747 ± 0.126` |
| `timing_all_histgb` | `0.658 ± 0.077` | `0.662 ± 0.071` | `0.715 ± 0.174` |
| `notebook05_static_features_histgb` | `0.641 ± 0.049` | `0.649 ± 0.049` | `0.662 ± 0.203` |

Conclusion : le notebook 05 avait identifié une vraie piste, mais son score était optimiste. PSO reste utile en validation stricte, car il bat la baseline `timing_all_histgb`, mais le gain est modéré. La performance réaliste de cette famille de méthode est plutôt autour de `0.67` F1 macro, pas `0.78`.

Cette variante reste intéressante pour le prototype, car elle n'utilise pas les features dépendantes du layout clavier. Elle est donc plus cohérente pour un test navigateur utilisé avec QWERTY ou AZERTY, mais elle ne doit pas être présentée comme un test autonome fiable.
